In [ ]:
# Cell 1 — Load everything
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))
from config import MODEL_PATH, METRICS_PATH, AVG_FRAUD_LOSS, ANALYST_REVIEW_COST
import joblib

bundle = joblib.load(MODEL_PATH)
rf_model     = bundle['random_forest']
lr_model     = bundle['logistic_regression']
iso_model    = bundle['isolation_forest']
feature_cols = bundle['feature_cols']
X_test       = bundle['X_test']
y_test       = bundle['y_test']
iso_scores   = bundle['iso_scores_test']

with open(METRICS_PATH) as f:
    metrics = json.load(f)

print(f'Models loaded: {list(bundle.keys())}')
print(f'Test set size: {len(y_test):,}')
print(f'Fraud in test: {int((y_test==1).sum())} ({int((y_test==1).sum())/len(y_test)*100:.3f}%)')
print(f'Metrics loaded from: {METRICS_PATH}')
print()
print('Key metrics at threshold 0.30:')
for k in ['pr_auc','roc_auc','precision','recall','f1','true_positives','false_positives','net_value']:
    print(f'  {k:<22} {metrics[k]}')

In [ ]:
# Cell 2 — Threshold sensitivity analysis
from sklearn.metrics import precision_score, recall_score, f1_score

y_true   = y_test.to_numpy()
rf_probs = rf_model.predict_proba(X_test[feature_cols])[:, 1]

thresholds = np.arange(0.10, 0.91, 0.05)
rows = []
for t in thresholds:
    y_pred = (rf_probs >= t).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    flagged = int(y_pred.sum())
    fraud_caught_pct = tp / max(1, int((y_true == 1).sum())) * 100
    net_val = tp * AVG_FRAUD_LOSS - fp * ANALYST_REVIEW_COST
    rows.append({'threshold': t, 'precision': prec, 'recall': rec, 'f1': f1,
                 'flagged': flagged, 'fraud_caught_pct': fraud_caught_pct, 'net_value': net_val})

sweep_df = pd.DataFrame(rows)

fig, ax1 = plt.subplots(figsize=(11, 6))
ax2 = ax1.twinx()

ax1.plot(sweep_df['threshold'], sweep_df['precision'], 'b-o', markersize=5, linewidth=2, label='Precision')
ax1.plot(sweep_df['threshold'], sweep_df['recall'],    'r-o', markersize=5, linewidth=2, label='Recall')
ax1.plot(sweep_df['threshold'], sweep_df['f1'],        'g--', markersize=5, linewidth=1.5, label='F1')
ax1.set_xlabel('Decision threshold')
ax1.set_ylabel('Precision / Recall / F1')
ax1.set_ylim(0, 1.05)

ax2.plot(sweep_df['threshold'], sweep_df['net_value'], 'k-^', markersize=5, linewidth=1.5,
         label='Net business value ($)', alpha=0.7)
ax2.set_ylabel('Net business value ($)')

ax1.axvline(0.30, color='red', linestyle='--', linewidth=2, label='Chosen threshold (0.30)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='lower left')

ax1.set_title('Precision-Recall-Value tradeoff across decision thresholds', fontsize=13)
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\nThreshold sweep summary:')
print(sweep_df.to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# Cell 3 — Model comparison table
from sklearn.metrics import auc, precision_recall_curve, roc_auc_score

THRESHOLD = 0.30

def _iso_probs(scores):
    shifted = -scores
    return (shifted - shifted.min()) / (shifted.max() - shifted.min() + 1e-12)

rf_probs  = rf_model.predict_proba(X_test[feature_cols])[:, 1]
lr_probs  = lr_model.predict_proba(X_test[feature_cols])[:, 1]
iso_probs = _iso_probs(iso_scores)
y_true    = y_test.to_numpy()
n_fraud   = int((y_true == 1).sum())

def model_row(name, probs, threshold):
    y_pred = (probs >= threshold).astype(int)
    tp = int(((y_pred==1)&(y_true==1)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    prec_val = precision_score(y_true, y_pred, zero_division=0)
    rec_val  = recall_score(y_true, y_pred, zero_division=0)
    f1_val   = f1_score(y_true, y_pred, zero_division=0)
    p_curve, r_curve, _ = precision_recall_curve(y_true, probs)
    pr_auc_val = auc(r_curve, p_curve)
    fraud_caught_pct = tp / max(1, n_fraud) * 100
    net_val = tp * AVG_FRAUD_LOSS - fp * ANALYST_REVIEW_COST
    return {
        'Model': name,
        'Precision': round(prec_val, 3),
        'Recall': round(rec_val, 3),
        'F1': round(f1_val, 3),
        'PR-AUC': round(pr_auc_val, 3),
        'Fraud Caught %': round(fraud_caught_pct, 1),
        'False Alarm Count': fp,
        'Net Business Value': round(net_val, 2),
    }

comparison = pd.DataFrame([
    model_row('Random Forest',       rf_probs,  THRESHOLD),
    model_row('Logistic Regression', lr_probs,  THRESHOLD),
    model_row('Isolation Forest',    iso_probs, THRESHOLD),
])

print(f'Model comparison at threshold {THRESHOLD}:')
print(comparison.to_string(index=False))

In [ ]:
# Cell 4 — False negative analysis
THRESHOLD = 0.30
rf_probs = rf_model.predict_proba(X_test[feature_cols])[:, 1]
y_true   = y_test.to_numpy()

fraud_mask   = y_true == 1
fn_mask      = fraud_mask & (rf_probs < THRESHOLD)
caught_mask  = fraud_mask & (rf_probs >= THRESHOLD)

fn_df     = X_test[fn_mask].copy()
caught_df = X_test[caught_mask].copy()

analyze_cols = ['hour_of_day', 'night_flag', 'V14', 'amount_log', 'v_sum']
analyze_cols = [c for c in analyze_cols if c in fn_df.columns]

fn_means     = fn_df[analyze_cols].mean().rename('Missed fraud')
caught_means = caught_df[analyze_cols].mean().rename('Caught fraud')

comp_table = pd.concat([fn_means, caught_means], axis=1)
comp_table['Difference'] = (comp_table['Caught fraud'] - comp_table['Missed fraud']).round(4)

print(f'False negative analysis (threshold={THRESHOLD}):')
print(f'  Missed fraud cases:  {fn_mask.sum()}')
print(f'  Caught fraud cases:  {caught_mask.sum()}')
print()
print(comp_table.round(4).to_string())

**What makes fraud cases hard to catch:** Missed fraud cases tend to have lower `night_flag` rates (daytime transactions are harder to distinguish), lower `v_sum` deviation (their behavioral V-feature pattern is closer to normal transactions), and their V14 values are less extreme. These are "low-signal" fraud cases where the transaction pattern closely mimics legitimate behavior — they would require additional features (device fingerprint, IP geolocation, account history) to catch reliably.

In [ ]:
# Cell 5 — Model agreement scatter plot
rf_probs  = rf_model.predict_proba(X_test[feature_cols])[:, 1]
y_true    = y_test.to_numpy()

# Subsample for plot speed (full test set can be 56K points)
rng = np.random.RandomState(42)
n_plot = min(5000, len(y_true))
plot_idx = rng.choice(len(y_true), size=n_plot, replace=False)

rf_plot  = rf_probs[plot_idx]
iso_plot = iso_scores[plot_idx]
y_plot   = y_true[plot_idx]

legit_mask = y_plot == 0
fraud_mask = y_plot == 1

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(rf_plot[legit_mask],  iso_plot[legit_mask],  c='#4A90D9', alpha=0.3, s=8,  label='Legitimate')
ax.scatter(rf_plot[fraud_mask],  iso_plot[fraud_mask],  c='#E24B4A', alpha=0.8, s=30, label='Fraud', marker='*')

ax.axvline(0.30, color='grey', linestyle='--', linewidth=1.5)
ax.axhline(0,    color='grey', linestyle='--', linewidth=1.5)

ax.text(0.65, iso_plot.max() * 0.92, 'Both flag\n(highest confidence)', ha='center',
        fontsize=9, color='darkred',   fontweight='bold')
ax.text(0.65, iso_plot.min() * 0.85, 'Only RF flags',   ha='center', fontsize=9, color='#8B0000')
ax.text(0.10, iso_plot.max() * 0.92, 'Only IF flags',   ha='center', fontsize=9, color='darkorange')
ax.text(0.10, iso_plot.min() * 0.85, 'Both clear\n(lowest risk)', ha='center', fontsize=9, color='steelblue')

ax.set_xlabel('Random Forest fraud probability')
ax.set_ylabel('Isolation Forest anomaly score (more negative = more anomalous)')
ax.set_title('Model Agreement: Random Forest vs Isolation Forest', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## Cell 6 — Headline Numbers for Resume

Run `python src/models/evaluate.py` first, then the numbers below will be populated from `outputs/metrics.json`.

```
Resume Bullet Numbers — fill these into resume after running evaluate.py

PR-AUC:                    [see metrics.json → pr_auc]
Fraud caught (%):          [recall × 100]%
Precision at 0.30:         [precision × 100]%
Recall at 0.30:            [recall × 100]%
Net value per test set:    $[net_value]
False positive rate:       [false_positives / (false_positives + true_negatives) × 100]%
Value caught ($):          $[value_caught]
Value missed ($):          $[value_missed]
```

In [ ]:
# Cell 6b — Auto-populate from metrics.json
import json
from config import METRICS_PATH

with open(METRICS_PATH) as f:
    m = json.load(f)

fp_rate = m['false_positives'] / max(1, m['false_positives'] + m['true_negatives']) * 100

print('=== Resume Bullet Numbers ===')
print(f"PR-AUC:                    {m['pr_auc']:.3f}")
print(f"Fraud caught (%):          {m['recall']*100:.1f}%")
print(f"Precision at 0.30:         {m['precision']*100:.1f}%")
print(f"Recall at 0.30:            {m['recall']*100:.1f}%")
print(f"Net value per test set:    ${m['net_value']:,.2f}")
print(f"False positive rate:       {fp_rate:.1f}%")
print(f"Value caught ($):          ${m['value_caught']:,.2f}")
print(f"Value missed ($):          ${m['value_missed']:,.2f}")